# 第14章 Agent 入门

**操作手册** | Agent、ReAct 与工具调用

本手册对应文档：`docs/part3-agent/chapter14/index.md`  
本手册对应代码：`code/part3-agent/`

---


## 在云端运行本章

原文中的性能数字来自 Radeon 8060S（`gfx1151`）参考运行。云端 GPU 型号不同，运行单元以当前设备输出为准。

云端已提供 ROCm、PyTorch 和 Triton；执行单元会补充 Agent 所需的 Python 第三方包。

本章以源码阅读和工具调用为主，不运行性能 benchmark。


## 本章目标、前置知识与产物

本章是 Agent 篇的入口。我们沿用 [hello-agents](https://github.com/datawhalechina/hello-agents) 的概念节奏——**Agent = LLM + 工具 + 循环**——但把场景收窄到「GPU 算子优化」：目标是延迟/带宽，反馈是可复现的 benchmark 数字，而不是开放域闲聊。

学完本章，你应该能够：

- 用一句话解释 Agent = LLM + 工具 + 循环；
- 说明为什么算子优化天然适合 Agent；
- 读懂 ReAct / Reflection 在本书主循环里的落点；
- 分清「权威工具」与「自由工具」，以及性能数字只认谁。

对应代码：

```text
code/part3-agent/
├── kernel_optimize/          # Reflection / ReAct Agent
│   ├── agent.py              # 主循环
│   ├── tools.py              # 工具注册与权威裁决
│   └── prompts.py            # 优化 SOP
└── chapter14/                # 评测器（下一章工具后端）
```



## 14.1 什么是 LLM Agent

最简模型：

```text
Agent ≈ 大模型（决策） + 工具（行动） + 循环（观察→再决策）
```

- **LLM**：读当前状态，决定下一步（调哪个工具、改哪段 kernel、是否收尾）。
- **工具**：确定性程序。编译、对答案、计时、测峰值——返回值就是事实。
- **循环**：把工具观察写回对话历史，再问模型，直到收敛或触达步数上限。

这和「一次性让模型吐出更快代码」的差别在于：**模型不掌握真假**。快不快、对不对，只认工具返回。



## 14.2 为什么 Agent 适合算子优化

算子优化有三个 Agent 友好的特征：

| 特征 | 在本书里的落点 |
|---|---|
| 目标可量化 | 延迟中位数、配对改进比例、距峰值空间 |
| 行动可封装 | `compile_kernel` / `bench_kernel` / `profile_kernel` / `accept_candidate` / `measure_peak` |
| 反馈可验证 | PyTorch reference 做 oracle；warmup + GPU event + median/MAD 做计时 |

因此闭环可以写成：

```text
理解任务 → 测峰值 → baseline → profiling 定方向
  → 生成候选 → compile → bench → profile → accept → 反思迭代 → 报告
```

缺任何一环都会出问题：没有峰值就不知道「算到头没有」；没有正确性门禁就会把「快但错」当成成果；没有配对裁决就会被噪声忽悠。

第 17 章会看到一次真实跑通：在 Radeon 8060S（`gfx1151`）上，Agent 把故意朴素的 `vector_add` 从约 **0.705 ms** 推到 **0.322 ms（≈2.19×）**——每一步改进都来自工具账本，不是模型口头宣布。



## 14.3 ReAct 范式简介

本书主循环是 **ReAct（Reason → Act → Observe）**，外层再套 **Reflection（根据评测结果改策略）**。`kernel_optimize/agent.py` 的骨架如下：

```python
for step in range(1, max_steps + 1):
    message = llm.chat(history, tools=schema)   # Reason：决定是否调工具
    if not message.tool_calls:                  # 不再行动 → 尝试收尾
        return message.content
    for call in message.tool_calls:             # Act：compile/bench/profile/accept…
        observation = executor.call(...)        # 工具给出 Observation
        history.append(observation)             # 观察回灌，进入下一轮
```

教学上刻意保持这段循环「能一眼看完」：复杂统计与裁决都下沉到工具里，主循环不堆业务。

另有一层**完成护栏**：模型若只用文本邀请用户、或把提问写进「最终回答」，会立刻结束运行——因此 Agent 必须用 `ask_user` 提问，用工具干活，而不是空谈收尾。



## 14.4 工具调用（Tool Use）

工具用「名字 + JSON 参数」描述，经 function calling 交给模型。注册方式是字典（hello-agents 范式）：描述给模型看，函数本体按名调用。

```text
ToolExecutor.register(name, description, func, parameters_schema)
→ litellm tools=[]
→ 模型返回 tool_calls[{name, arguments}]
→ executor.call(name, args) → 字符串观察（内容为 JSON）
```

本书把工具分成两类（自由度光谱）：

1. **权威工具**（结果即事实）：`compile_kernel`、`bench_kernel`、`profile_kernel`、`accept_candidate`、`measure_peak`
2. **自由工具**（允许发挥）：`ask_user`、`convert_kernel`、`run_code`、`read_reference`

纪律写进系统提示：**性能数字只认权威工具**；禁止用 `run_code` 自测出一个「加速比」写进报告。晋升由 `accept_candidate` 单独完成。



## 14.5 本书 Agent 的边界

做：

- 给定算子规格 / baseline kernel，在本机 ROCm GPU 上迭代优化；
- 正确性优先，失败留痕，搜索与结论分离。

不做：

- 通用 IDE Agent / 自动写业务服务；
- 让 LLM 口头宣布「快了 3×」而不经评测器；
- 把某张卡的峰值常数写死进教程当真理（峰值必须 `measure_peak` 实测）。

执行语言上，当前评测器跑 **Triton**；CUDA/HIP C++ 需先 `convert_kernel` 成等价 Triton，再进入优化闭环。



## 14.6 和 hello-agents 的关系

| | hello-agents | 本篇 hello-gpu |
|---|---|---|
| 教学目标 | 通用 Agent 范式 | 把「算子优化闭环」讲透并跑通 |
| 工具 | 示例级 | 可信计时 + 配对裁决 + Roofline |
| 真假来源 | 视任务而定 | **确定性评测器**（第 15 章） |

零基础建议先浏览 hello-agents 的 ReAct / Tool Use 章节；有 Agent 基础可直接进入第 15 章看工具如何「硬起来」。



## Execution

### 步骤1：定位仓库根目录


In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        for repo in (candidate, candidate / "hello-gpu"):
            if (
                (repo / "code/part3-agent/kernel_optimize/agent.py").is_file()
                and (repo / "notebooks/part3-agent").is_dir()
            ):
                return repo
    raise FileNotFoundError("未找到 hello-gpu 仓库；请从仓库目录或其父目录打开本 Notebook。")


REPO_ROOT = find_repo_root()
PART3_ROOT = REPO_ROOT / "code/part3-agent"
FIXTURE_ROOT = PART3_ROOT / "chapter15/fixtures/vector_add"
if str(PART3_ROOT) not in sys.path:
    sys.path.insert(0, str(PART3_ROOT))

print(f"仓库根目录: {REPO_ROOT}")
print(f"代码目录: {PART3_ROOT}")
print(f"Python: {sys.executable}")


### 步骤2：准备 Agent 依赖

本章使用 `litellm`、`prompt_toolkit` 等第三方包。PyTorch、Triton 和 ROCm 使用云端已有环境。


In [ ]:
requirements = {
    "litellm": "litellm>=1.50",
    "pydantic": "pydantic>=2.0",
    "prompt_toolkit": "prompt-toolkit>=3.0",
    "fastapi": "fastapi>=0.100",
    "orjson": "orjson>=3.0",
}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]

if missing:
    command = [sys.executable, "-m", "pip", "install", *missing]
    print("安装缺失依赖:", " ".join(missing))
    subprocess.run(command, check=True)
    importlib.invalidate_caches()
else:
    print("Agent 依赖已就绪")


### 步骤3：定位 ReAct 主循环

读取 `kernel_optimize/agent.py` 中的 `run_agent`，查看模型调用、工具路由和 Observation 回灌分别落在哪一行。


In [ ]:
import inspect
from kernel_optimize.agent import run_agent

source_lines, first_line = inspect.getsourcelines(run_agent)
markers = ("llm.chat", "tool_calls", "executor.call", '"role": "tool"')
for offset, line in enumerate(source_lines):
    if any(marker in line for marker in markers):
        print(f"{first_line + offset:>4}: {line.rstrip()}")


### 步骤4：查看工具 schema

`build_tools` 返回 LiteLLM function calling 使用的 JSON schema。下面列出工具名，并调用 `get_environment` 查看一次真实 Observation。


In [ ]:
import tempfile
from kernel_optimize.tools import Workspace, build_tools

with tempfile.TemporaryDirectory(prefix="hello-gpu-ch14-") as temporary:
    workspace = Workspace(Path(temporary))
    executor, schema = build_tools(workspace, batch=True)
    tool_names = [item["function"]["name"] for item in schema]
    observation = executor.call("get_environment", {})

print("注册工具:")
for name in tool_names:
    print(f"  - {name}")
print("\nget_environment Observation:")
print(json.dumps(json.loads(observation), ensure_ascii=False, indent=2))


## Expected Output / Interpretation

主循环定位结果应包含 `llm.chat`、`tool_calls`、`executor.call` 和 `role=tool`；工具列表应包含 `compile_kernel`、`bench_kernel`、`profile_kernel`、`accept_candidate` 与 `measure_peak`。

`get_environment` 返回当前设备、GPU 架构、ROCm 和 PyTorch 版本。它是一次工具 Observation，不是性能测试结果。


## Pass Criteria

1. 能从源码指出 ReAct 循环的四个关键落点；
2. 能读出工具 schema 中的工具名和参数结构；
3. 能区分权威工具与自由工具，并说明性能数字只认权威工具。


## 本章小结

- Agent = LLM + 工具 + 循环；在算子优化里，工具返回值就是 ground truth。
- 本书主骨架是 ReAct，外加 Reflection；主循环保持短小可读。
- 「LLM 负责理解，代码负责相信什么」——后续三章都围着这句话展开。



## 延伸阅读

- [hello-agents](https://github.com/datawhalechina/hello-agents)
- [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629)
- 本仓库 `code/part3-agent/HANDOFF.md`（方法论交接）
